# Spremljanje vseh Planet naročnin

Ta zvezek izpiše **vse vaše naročnine Planet Subscriptions API** in jih spremlja v pregledni tabeli.

Osredotoča se na:
- izpis vseh naročnin, do katerih imate dostop,
- pridobivanje trenutnega stanja posamezne naročnine,
- pridobivanje **povzetka** števcev za vsako naročnino,
- prikaz naročnin, ki so še vedno v ozadju v obdelavi ali backfill fazi,
- možnost periodičnega osveževanja, dokler se aktivno delo ne umiri.

## Kaj uporablja
- `GET /subscriptions/v1/subscriptions` za seznam naročnin,
- `GET /subscriptions/v1/subscriptions/{id}` za podrobnosti,
- `GET /subscriptions/v1/subscriptions/{id}/summary` za spremljanje števcev, kot so `queued`, `processing`, `success` in `failed`.

## Opombe
- Ta zvezek je samo za branje. Ne ustvarja, ne posodablja, ne ustavlja in ne preklicuje naročnin.
- Naročnino lahko štejemo kot "trenutno mirno", ko `queued == 0` in `processing == 0`.
- Če API vrne `backfill_complete`, je to najmočnejši signal, da je zgodovinski backfill zaključen.


In [1]:
import os
import json
import time
from typing import List, Optional

import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

In [2]:
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = 'PASTE_YOUR_API_KEY_HERE'

BASE = "https://api.planet.com/subscriptions/v1"
PAGE_SIZE = 100  # po potrebi prilagodi

auth = HTTPBasicAuth(API_KEY, "")
session = requests.Session()
session.auth = auth
session.headers.update({"Accept": "application/json"})

In [3]:
# Output folder
output_folder = "rezultati"
# Preveri ali obstaja mapa rezultati, če ne, jo ustvari
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
# Shranjevanje stanja v CSV datoteko, v mapi output_folder
output_csv = os.path.join(output_folder, "planet_narocnine_monitor.csv")

## Pomožne funkcije

In [4]:
def get_json(url: str, params: Optional[dict] = None, timeout: int = 120) -> dict:
    response = session.get(url, params=params, timeout=timeout)
    try:
        data = response.json()
    except ValueError:
        response.raise_for_status()
        raise RuntimeError(f"Ne-JSON odziv za {url}: {response.text[:500]}")
    response.raise_for_status()
    return data


def extract_items(payload: dict) -> List[dict]:
    # Podpora pogostim oblikam API ovojnice
    for key in ("subscriptions", "items", "data"):
        if isinstance(payload.get(key), list):
            return payload[key]
    if isinstance(payload, list):
        return payload
    return []


def extract_next_link(payload: dict) -> Optional[str]:
    # Podpora več pogostim slogom straničenja
    links = payload.get("_links") or payload.get("links") or {}
    if isinstance(links, dict):
        nxt = links.get("_next") or links.get("next")
        if isinstance(nxt, dict):
            return nxt.get("href")
        if isinstance(nxt, str):
            return nxt

    # Nadomestno straničenje s tokenom
    next_token = payload.get("next") or payload.get("next_page_token") or payload.get("page_marker")
    if next_token:
        if str(next_token).startswith("http"):
            return str(next_token)
    return None


def list_all_subscriptions(page_size: int = 100, verbose: bool = True) -> List[dict]:
    url = BASE
    params = {"page_size": page_size}
    all_items: List[dict] = []

    while True:
        payload = get_json(url, params=params)
        items = extract_items(payload)
        all_items.extend(items)

        if verbose:
            print(f"Prenesenih {len(items)} naročnin; skupaj do zdaj: {len(all_items)}")

        next_link = extract_next_link(payload)
        if not next_link:
            break

        if next_link.startswith("http"):
            url = next_link
            params = None
        else:
            # Če API vrne relativno pot ali token, ga obravnavamo previdno
            url = next_link if next_link.startswith("/") else BASE
            params = None if next_link.startswith("/") else {"page_marker": next_link, "page_size": page_size}

    return all_items


def get_subscription(subscription_id: str) -> dict:
    return get_json(f"{BASE}/{subscription_id}")


def get_summary(subscription_id: str) -> dict:
    return get_json(f"{BASE}/{subscription_id}/summary")


def deep_get_collection_id(obj):
    if isinstance(obj, dict):
        hosting = obj.get("hosting")
        if isinstance(hosting, dict):
            params = hosting.get("parameters", {})
            if isinstance(params, dict) and params.get("collection_id"):
                return params["collection_id"]
        for value in obj.values():
            found = deep_get_collection_id(value)
            if found:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = deep_get_collection_id(value)
            if found:
                return found
    return None


def summarize_one(subscription: dict, details: Optional[dict] = None, summary: Optional[dict] = None) -> dict:
    sid = subscription.get("id")
    name = subscription.get("name") or subscription.get("title") or ""
    status = (details or subscription).get("status")
    created = subscription.get("created") or (details or {}).get("created")
    updated = subscription.get("updated") or (details or {}).get("updated")
    collection_id = deep_get_collection_id(details or subscription)

    result_counts = {}
    if isinstance(summary, dict):
        result_counts = summary.get("results", {}) or {}

    queued = result_counts.get("queued", 0)
    processing = result_counts.get("processing", 0)
    success = result_counts.get("success", 0)
    failed = result_counts.get("failed", 0)
    created_results = result_counts.get("created", 0)

    backfill_complete = None
    if isinstance(details, dict):
        backfill_complete = details.get("backfill_complete")

    active_work = (queued or 0) + (processing or 0)

    if backfill_complete:
        state_hint = "backfill zaključen"
    elif active_work > 0:
        state_hint = "v delu"
    elif status in {"running", "active", "completed"}:
        state_hint = "mirno/stabilno"
    else:
        state_hint = "preveri stanje"

    return {
        "id": sid,
        "name": name,
        "status": status,
        "state_hint": state_hint,
        "created": created,
        "updated": updated,
        "backfill_complete": backfill_complete,
        "collection_id": collection_id,
        "queued": queued,
        "processing": processing,
        "success": success,
        "failed": failed,
        "created_results": created_results,
        "active_work": active_work,
    }

## Izpis vseh naročnin

In [5]:
subscriptions = list_all_subscriptions(page_size=PAGE_SIZE, verbose=True)
print(f"Skupno najdenih naročnin: {len(subscriptions)}")

# Po želji: izpis prvega surovega zapisa za pregled oblike odgovora
if subscriptions:
    print(json.dumps(subscriptions[0], indent=2)[:2000])

Prenesenih 1 naročnin; skupaj do zdaj: 1
Skupno najdenih naročnin: 1
{
  "name": "Kras",
  "source": {
    "type": "catalog",
    "parameters": {
      "item_types": [
        "PSScene"
      ],
      "asset_types": [
        "ortho_analytic_8b_sr",
        "ortho_analytic_8b_xml",
        "ortho_udm2"
      ],
      "filter": {
        "type": "StringInFilter",
        "config": [
          "standard"
        ],
        "field_name": "quality_category"
      },
      "publishing_stages": [
        "standard",
        "finalized"
      ],
      "time_range_type": "acquired",
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              13.45946938,
              45.66358961
            ],
            [
              14.00318114,
              45.66358961
            ],
            [
              14.00318114,
              45.92255159
            ],
            [
              13.45946938,
              45.92255159
            ],
      

## Izdelava tabele spremljanja

Ta del za vsako naročnino zahteva podrobni zapis in povzetek.


In [6]:
rows = []

for i, sub in enumerate(subscriptions, start=1):
    sid = sub.get("id")
    if not sid:
        continue

    try:
        details = get_subscription(sid)
    except Exception as e:
        details = {"status": f"napaka_podrobnosti: {e}"}

    try:
        summary = get_summary(sid)
    except Exception as e:
        summary = {"results": {}, "summary_error": str(e)}

    row = summarize_one(sub, details=details, summary=summary)
    rows.append(row)

    print(f"[{i}/{len(subscriptions)}] {row['name'] or sid} -> stanje={row['status']}, aktivno_delo={row['active_work']}")

df = pd.DataFrame(rows)

if not df.empty:
    df = df.sort_values(
        by=["active_work", "status", "updated"],
        ascending=[False, True, False],
        na_position="last"
    ).reset_index(drop=True)

df

[1/1] Kras -> stanje=running, aktivno_delo=0


,id,name,status,state_hint,created,updated,backfill_complete,collection_id,queued,processing,success,failed,created_results,active_work
0,747d33ba-88c6-40b2-9c94-3021d65e360a,Kras,running,mirno/stabilno,2026-03-27T13:42:15.671642Z,2026-03-27T13:42:47.531844Z,None,69eeb2e0-8fa7-4a47-adab-e300cdc39a2f,0,0,6762,0,0,0


## Dodatni pogledi

In [7]:
# Naročnine, ki delujejo kot aktivne
busy_df = df[(df["active_work"] > 0) | (df["backfill_complete"].isna())].copy() if not df.empty else pd.DataFrame()
busy_df

,id,name,status,state_hint,created,updated,backfill_complete,collection_id,queued,processing,success,failed,created_results,active_work
0,747d33ba-88c6-40b2-9c94-3021d65e360a,Kras,running,mirno/stabilno,2026-03-27T13:42:15.671642Z,2026-03-27T13:42:47.531844Z,None,69eeb2e0-8fa7-4a47-adab-e300cdc39a2f,0,0,6762,0,0,0


In [8]:
# Strnjen operativni pogled
compact_cols = [
    "name", "status", "state_hint", "backfill_complete",
    "queued", "processing", "success", "failed",
    "collection_id", "updated"
]

compact_df = df[compact_cols].copy() if not df.empty else pd.DataFrame()
compact_df

,name,status,state_hint,backfill_complete,queued,processing,success,failed,collection_id,updated
0,Kras,running,mirno/stabilno,None,0,0,6762,0,69eeb2e0-8fa7-4a47-adab-e300cdc39a2f,2026-03-27T13:42:47.531844Z


## Izvoz trenutne tabele

Uporabno, če želite stanje spremljanja pregledovati tudi zunaj zvezka.


In [9]:
df.to_csv(output_csv, index=False)
print(f"Datoteka zapisana: {output_csv}")

Datoteka zapisana: rezultati\planet_narocnine_monitor.csv


## Odpravljanje težav

- Če izpis vrne nič naročnin, preverite, ali ima API ključ dostop do Subscriptions API.
- Če endpoint za povzetek odpove pri nekaterih naročninah, preverite surov zapis z `get_subscription(subscription_id)`.
- Če se straničenje v vašem računu obnaša drugače, preverite surov odgovor in po potrebi prilagodite `extract_items()` / `extract_next_link()`.
- Če je `backfill_complete` prisoten in ni `null`, ga obravnavajte kot najboljši signal, da je zgodovinski zajem zaključen.
